# 09 - Model Station (3 segments/jour) CatBoost Dual-Model

Objectif: predire la demande par station sur 3 segments par jour:
- `00:00-05:59` (night)
- `06:00-15:59` (day)
- `16:00-23:59` (evening)

Strategie modele:
- 1 modele pour stations a fort volume (high_volume)
- 1 modele pour stations a faible volume (low_volume)


In [1]:
import gc
import numpy as np
import pandas as pd
from pathlib import Path

from bike_demand_forecasting.utils import get_paths
from bike_demand_forecasting.features import (
    add_holiday_feature_us,
    add_lag_features_by_station,
    add_rolling_features_by_station,
    add_sin_cos_features,
    time_train_test_split,
    make_cv_splits_by_date
)
from bike_demand_forecasting.metrics import compute_metrics, plot_actual_vs_pred, smape, bias

from sklearn.model_selection import GridSearchCV
from sklearn.metrics import make_scorer
from catboost import CatBoostRegressor

pd.set_option("display.max_columns", None)

In [2]:
# Load and define path data merged 2024 2025
paths = get_paths()
DATA_DIR_MERGED = paths["DATA_DIR_MERGED"]
data_merged_path = DATA_DIR_MERGED / "all_merged.csv"
data_merged_path

PosixPath('/mnt/c/Users/33760/Desktop/bike-demand-forecasting/data/merged/all_merged.csv')

In [3]:
# Read csv with chunks
CHUNKSIZE = 500_000
# Select usefull columns
cols_select = ["started_at", "start_station_id"]
# empty list to retrieve dat
list_station_demand = []

# Define reader
reader = pd.read_csv(
    data_merged_path,
    usecols=cols_select,
    dtype={
        "started_at": "string",
        "start_station_id": "string",
    },
    chunksize=CHUNKSIZE,
    low_memory=False,
)

## LOOP to retrieve data
for chunk in reader:
    # Clean date column
    s = chunk["started_at"].str.strip().str.replace(r"\.\d+$", "", regex=True)
    # Convert column started_at to datetime
    dt = pd.to_datetime(s, format="%Y-%m-%d %H:%M:%S", errors="coerce")
    # Convert start station_id to numerics
    sid = pd.to_numeric(chunk["start_station_id"].str.strip(), errors="coerce").astype("Int32")
    # Health check (boolean mask :NaN)
    valid = dt.notna() & sid.notna()
    if not valid.any():
        continue
    
    # Apply boolean mask nan values
    dtv = dt.loc[valid]
    sidv = sid.loc[valid]

    # extract hour (0 to 23) from datetime column
    hour = dtv.dt.hour
    # Define the segment column from hour column
    segment_id = np.select([hour < 6, hour < 16], [0, 1], default=2).astype("int8")
    segment_start_hour = np.select([hour < 6, hour < 16], [0, 6], default=16).astype("int8")
    # Align each timestamp to the start of its 3-segment slot (00:00, 06:00, or 16:00)
    date_segment = dtv.dt.normalize() + pd.to_timedelta(segment_start_hour, unit="h")

    # Build station-segment demand rows for the current chunk
    station_demand = pd.DataFrame({
        "start_station_id": sidv.to_numpy(),
        "date": date_segment.to_numpy(),
        "segment_id": segment_id,
    })

    # Aggregate trips into demand counts per station/date/segment
    station_demand = (
        station_demand.groupby(["start_station_id", "date", "segment_id"], as_index=False)
        .size()
        .rename(columns={"size": "y_station"})
    )
    list_station_demand.append(station_demand)


In [4]:
# Re-aggregate identical station/date/segment keys that may be split across chunks
df_station = (
    pd.concat(list_station_demand, ignore_index=True)
    .groupby(["start_station_id", "date", "segment_id"], as_index=False)["y_station"]
    .sum()
    .sort_values(["start_station_id", "date", "segment_id"])
    .reset_index(drop=True)
)

# Cast demand counts to int32 to reduce memory while preserving safe range
df_station["y_station"] = df_station["y_station"].astype("int32")

In [5]:
df_station.head()

,start_station_id,date,segment_id,y_station
0,30200,2024-01-01 00:00:00,0,1
1,30200,2024-01-01 06:00:00,1,4
2,30200,2024-01-02 06:00:00,1,9
3,30200,2024-01-02 16:00:00,2,3
4,30200,2024-01-03 00:00:00,0,1


In [5]:
## Build a complete station x day x segment grid, then left-join observed demand
## so missing slots are explicitly filled with y_station = 0.

# List unique station ids to build a complete panel index
stations = pd.DataFrame({
    "start_station_id": df_station["start_station_id"].drop_duplicates().sort_values()
})

# Build the full daily date range covered by the dataset
day_range = pd.date_range(
    df_station["date"].min().normalize(),
    df_station["date"].max().normalize(),
    freq="D",
)
days = pd.DataFrame({"day": day_range})

# Define the three daily segments and their start hours
segment_slots = pd.DataFrame({
    "segment_id": pd.Series([0, 1, 2], dtype="int8"),
    "segment_start_hour": pd.Series([0, 6, 16], dtype="int8"),
})

# Build all day x segment combinations and map them to segment start timestamps
day_segment_slots = days.merge(segment_slots, how="cross")
day_segment_slots["date"] = day_segment_slots["day"] + pd.to_timedelta(day_segment_slots["segment_start_hour"], unit="h")
day_segment_slots = day_segment_slots[["date", "segment_id"]]

# Build the full station x (day,segment) grid
full_index = stations.merge(day_segment_slots, how="cross")

# Left-join observed demand onto the full grid (missing slots remain NaN)
panel = full_index.merge(
    df_station,
    on=["start_station_id", "date", "segment_id"],
    how="left",
)

# Fill missing demand with 0 and cast to compact integer dtype
panel["y_station"] = panel["y_station"].fillna(0).astype("int16")

# Flag rows that were missing in source data and filled with zero
panel["is_filled_zero"] = (panel["y_station"] == 0).astype("int8")

# Keep a stable sorted panel for downstream feature engineering
df_station_halfday = panel.sort_values(["start_station_id", "date", "segment_id"]).reset_index(drop=True)

In [6]:
# Free memory
for _v in ["chunk", "reader", "list_station_demand", "df_station", "panel"]:
    if _v in globals():
        del globals()[_v]
gc.collect()

250

In [7]:
# Time features for 3-segment granularity
df_station_halfday["year"] = df_station_halfday["date"].dt.year.astype("int16")
df_station_halfday["month"] = df_station_halfday["date"].dt.month.astype("int8")
df_station_halfday["month_num"] = df_station_halfday["month"]
df_station_halfday["dayofweek"] = df_station_halfday["date"].dt.dayofweek.astype("int8")
df_station_halfday["dayofyear"] = df_station_halfday["date"].dt.dayofyear.astype("int16")
df_station_halfday["hour"] = df_station_halfday["date"].dt.hour.astype("int8")  # 0, 6, 16

df_station_halfday["segment_id"] = df_station_halfday["segment_id"].astype("int8")
df_station_halfday["is_weekend"] = (df_station_halfday["dayofweek"] >= 5).astype("int8")

In [8]:
# Segment helper columns for analysis readability
segment_name_map = {0: "night_00_05", 1: "day_06_15", 2: "evening_16_23"}
df_station_halfday["segment_name"] = df_station_halfday["segment_id"].map(segment_name_map)

assert df_station_halfday["segment_id"].isin([0, 1, 2]).all()

In [9]:
# Feature engineering (holiday, lags, rolling, cyclical)

df_feat = add_holiday_feature_us(df_station_halfday, dt_col="date")

# 3 segments/day => weekly lag ~= 21 steps, bi-weekly ~= 42
df_feat = add_lag_features_by_station(
    df_feat,
    target_col="y_station",
    lags=(1, 2, 3, 21, 42),
    station_col="start_station_id",
    time_col="date",
)

df_feat = add_rolling_features_by_station(
    df_feat,
    target_col="y_station",
    windows=(3, 21, 42),
    shift_steps=1,
    station_col="start_station_id",
    time_col="date",
)

df_feat = add_sin_cos_features(df_feat)


In [10]:
# Memory-friendly dtypes
num_cols = [
    "lag_1", "lag_2", "lag_3", "lag_21", "lag_42",
    "roll_mean_3", "roll_std_3", "roll_mean_21", "roll_std_21", "roll_mean_42", "roll_std_42",
    "dayofw_sin", "dayofw_cos", "dayofy_sin", "dayofy_cos",
    "month_sin", "month_cos", "hour_sin", "hour_cos", "year",
]

for c in num_cols:
    if c in df_feat.columns:
        df_feat[c] = df_feat[c].astype("float32")

for c in ["is_holiday", "is_weekend", "segment_id"]:
    if c in df_feat.columns:
        df_feat[c] = df_feat[c].astype("int8")

# Drop rows where lag/rolling are NaN
lag_roll_cols = [c for c in df_feat.columns if c.startswith(("lag_", "roll_"))]
df_feat = df_feat.dropna(subset=lag_roll_cols).reset_index(drop=True)

In [11]:
# Split the feature table into time-ordered train/test sets and return the aligned cutoff timestamp
X_train, y_train, X_test, y_test, cutoff = time_train_test_split(df_feat)

In [12]:
# Reset indices after temporal split to keep row alignment clean
# (required for train CV indices; optional but harmless for test)
X_train = X_train.reset_index(drop=True)
y_train = y_train.reset_index(drop=True)

X_test = X_test.reset_index(drop=True)
y_test = y_test.reset_index(drop=True)

In [13]:
# CatBoost setup
# Keep date for plotting but exclude non-model columns from model matrix
feature_cols_cb = [c for c in X_train.columns if c not in ["date", "segment_name"]]

# Include segment_id in categorical columns
cat_cols = ["start_station_id", "is_holiday", "is_weekend", "segment_id"]

# Selection BEFORE model fitting (no pipeline/lambda)
X_train_model = X_train[feature_cols_cb].copy()
X_test_model = X_test[feature_cols_cb].copy()

# Split stations by cumulative train volume (80% volume => high stations)
station_volume_train = (
    pd.DataFrame({"start_station_id": X_train["start_station_id"], "y_station": y_train})
    .groupby("start_station_id", as_index=False)["y_station"]
    .sum()
    .sort_values("y_station", ascending=False)
    .reset_index(drop=True)
)

# Compute cumulative volume share after sorting stations by descending train demand
station_volume_train["cum_share"] = station_volume_train["y_station"].cumsum() / station_volume_train["y_station"].sum()

# High-volume stations = stations covering the first 80% of cumulative train volume
high_stations = set(
    station_volume_train.loc[station_volume_train["cum_share"] <= 0.80, "start_station_id"]
    .astype(int)
    .tolist()
)

# Safety fallback: ensure high_stations is never empty
if len(high_stations) == 0:
    high_stations = {int(station_volume_train["start_station_id"].iloc[0])}

# Low-volume stations = remaining train stations
all_train_stations = set(station_volume_train["start_station_id"].astype(int).tolist())
low_stations = all_train_stations - high_stations

# Quick diagnostics on split sizes and effective covered volume
print(f"High stations: {len(high_stations)} | Low stations: {len(low_stations)}")
print(
    f"High volume share: "
    f"{station_volume_train.loc[station_volume_train['start_station_id'].astype(int).isin(high_stations), 'y_station'].sum() / station_volume_train['y_station'].sum():.2%}"
)

High stations: 247 | Low stations: 609
High volume share: 79.88%


In [14]:
# High vs Low station counts (from train split)
print(f"Nb stations high_volume: {len(high_stations)}")
print(f"Nb stations low_volume: {len(low_stations)}")
print(f"Nb stations total: {len(high_stations) + len(low_stations)}")

high_volume_share = station_volume_train.loc[
    station_volume_train["start_station_id"].astype(int).isin(high_stations),
    "y_station"
].sum() / station_volume_train["y_station"].sum()

print(f"High volume share (train): {high_volume_share:.2%}")
print(f"Low volume share (train): {(1 - high_volume_share):.2%}")

print("X_train_model.shape:", X_train_model.shape)
print("X_test_model.shape :", X_test_model.shape)

mask_train_high = X_train["start_station_id"].astype(int).isin(high_stations)
mask_train_low = X_train["start_station_id"].astype(int).isin(low_stations)

print("X_train_high.shape:", X_train_model.loc[mask_train_high].shape)
print("X_train_low.shape :", X_train_model.loc[mask_train_low].shape)

Nb stations high_volume: 247
Nb stations low_volume: 609
Nb stations total: 856
High volume share (train): 79.88%
Low volume share (train): 20.12%
X_train_model.shape: (1474032, 29)
X_test_model.shape : (367224, 29)
X_train_high.shape: (425334, 29)
X_train_low.shape : (1048698, 29)


In [15]:
base_cb = CatBoostRegressor(
    loss_function="Poisson",
    eval_metric="Poisson",
    random_seed=42,
    thread_count=1,
    verbose=100,
    allow_writing_files=False,
)

# High-volume model: slightly higher capacity
param_grid_high = {
    "depth": [8],
    "learning_rate": [0.05],
    "iterations": [800],
    "l2_leaf_reg": [3],
}

# Low-volume model: more regularized
param_grid_low = {
    "depth": [6],
    "learning_rate": [0.03],
    "iterations": [500],
    "l2_leaf_reg": [8],
}

scoring = {
    "mae": "neg_mean_absolute_error",
    "smape": make_scorer(smape, greater_is_better=False),
    "bias": make_scorer(bias, greater_is_better=False),
}


In [ ]:
# Train high-volume model
mask_train_high = X_train["start_station_id"].astype(int).isin(high_stations)
X_train_high_full = X_train.loc[mask_train_high].reset_index(drop=True)
y_train_high = y_train.loc[mask_train_high].reset_index(drop=True)

X_train_high = X_train_high_full[feature_cols_cb].copy()

cv_splits_high = make_cv_splits_by_date(X_train_high_full, n_splits=3, test_size=3 * 30, gap=42)

grid_high = GridSearchCV(
    estimator=base_cb,
    param_grid=param_grid_high,
    cv=cv_splits_high,
    scoring=scoring,
    refit="mae",
    n_jobs=4,
    pre_dispatch="2*n_jobs",
    verbose=100,
)

grid_high.fit(X_train_high, y_train_high, cat_features=cat_cols)
best_model_high = grid_high.best_estimator_

print("High model best params:", grid_high.best_params_)

Fitting 3 folds for each of 1 candidates, totalling 3 fits
[CV 2/3; 1/1] START depth=8, iterations=800, l2_leaf_reg=3, learning_rate=0.05..
[CV 1/3; 1/1] START depth=8, iterations=800, l2_leaf_reg=3, learning_rate=0.05..
0:	learn: -1.1536731	total: 1.31s	remaining: 17m 27s
0:	learn: -1.1308544	total: 1.35s	remaining: 17m 59s
[CV 3/3; 1/1] START depth=8, iterations=800, l2_leaf_reg=3, learning_rate=0.05..
0:	learn: -1.1832240	total: 1.68s	remaining: 22m 20s


In [ ]:
# Train low-volume model
mask_train_low = X_train["start_station_id"].astype(int).isin(low_stations)
assert mask_train_low.any(), "No low-volume stations found in train split."

X_train_low_full = X_train.loc[mask_train_low].reset_index(drop=True)
y_train_low = y_train.loc[mask_train_low].reset_index(drop=True)
X_train_low = X_train_low_full[feature_cols_cb].copy()

cv_splits_low = make_cv_splits_by_date(X_train_low_full, n_splits=3, test_size=3 * 30, gap=42)

grid_low = GridSearchCV(
    estimator=base_cb,
    param_grid=param_grid_low,
    cv=cv_splits_low,
    scoring=scoring,
    refit="mae",
    n_jobs=4,
    pre_dispatch=4,
    verbose=100,
)

grid_low.fit(X_train_low, y_train_low, cat_features=cat_cols)
best_model_low = grid_low.best_estimator_

print("Low model best params:", grid_low.best_params_)


## Evaluation Global


In [ ]:
# Global metrics (dual-model routing)

# TRAIN routing
mask_train_high = X_train["start_station_id"].astype(int).isin(high_stations).to_numpy()
mask_train_low = X_train["start_station_id"].astype(int).isin(low_stations).to_numpy()

y_train_pred = np.zeros(len(X_train), dtype=float)
if mask_train_high.any():
    y_train_pred[mask_train_high] = best_model_high.predict(X_train_model.loc[mask_train_high])
if mask_train_low.any():
    y_train_pred[mask_train_low] = best_model_low.predict(X_train_model.loc[mask_train_low])

# TEST routing
mask_test_high = X_test["start_station_id"].astype(int).isin(high_stations).to_numpy()
mask_test_low = X_test["start_station_id"].astype(int).isin(low_stations).to_numpy()
mask_test_unknown = ~(mask_test_high | mask_test_low)

y_test_pred = np.zeros(len(X_test), dtype=float)
if mask_test_high.any():
    y_test_pred[mask_test_high] = best_model_high.predict(X_test_model.loc[mask_test_high])
if mask_test_low.any():
    y_test_pred[mask_test_low] = best_model_low.predict(X_test_model.loc[mask_test_low])
if mask_test_unknown.any():
    # Fallback for unseen stations in TEST
    y_test_pred[mask_test_unknown] = best_model_high.predict(X_test_model.loc[mask_test_unknown])

print(f"Unknown TEST-station rows routed to high model: {int(mask_test_unknown.sum())}")

compute_metrics(y_train, y_train_pred, "Train 3-segment (dual)")
compute_metrics(y_test, y_test_pred, "Test 3-segment (dual)")

plot_actual_vs_pred(
    X_test,
    y_test,
    y_test_pred,
    date_col="date",
    aggregate=True,
    title="Actual vs Predicted Demand (3-segment dual-model, all stations, TEST)",
)


In [ ]:
# Shared aligned test objects for subset analyses (same style as notebook 07)
X_test_eval = X_test.reset_index(drop=True)
y_true = y_test.reset_index(drop=True)
y_pred = pd.Series(y_test_pred).reset_index(drop=True)

assert len(X_test_eval) == len(y_true) == len(y_pred)

# Top/bottom station ids computed from TRAIN volume
station_volume_train = (
    pd.DataFrame({"start_station_id": X_train["start_station_id"], "y_station": y_train})
    .groupby("start_station_id", as_index=False)["y_station"]
    .sum()
)

top_station_id = station_volume_train.sort_values("y_station", ascending=False)["start_station_id"].iloc[0]
bottom_station_id = station_volume_train.sort_values("y_station", ascending=True)["start_station_id"].iloc[0]

assert top_station_id != bottom_station_id


## Test (y_true > 3)


In [ ]:
mask_pos = y_true > 0

X_test_pos = X_test_eval.loc[mask_pos]
y_test_pos = y_true.loc[mask_pos]
y_test_pred_pos = y_pred.loc[mask_pos]

compute_metrics(y_test_pos, y_test_pred_pos, "Test (y_true > 3)", mask_pos)

plot_actual_vs_pred(
    X_test_pos,
    y_test_pos,
    y_test_pred_pos,
    date_col="date",
    aggregate=True,
    title="Actual vs Predicted (Test, y_true > 3)",
)


## Top Station (Test)


In [ ]:
mask_top = X_test_eval["start_station_id"] == top_station_id

if mask_top.any():
    X_test_top = X_test_eval.loc[mask_top]
    y_test_top = y_true.loc[mask_top]
    y_test_pred_top = y_pred.loc[mask_top]

    compute_metrics(y_test_top, y_test_pred_top, "Top station (Test)", mask_top)

    plot_actual_vs_pred(
        X_test_top,
        y_test_top,
        y_test_pred_top,
        date_col="date",
        aggregate=True,
        title=f"Actual vs Predicted (Top station_id={top_station_id}, TEST)",
    )
else:
    print(f"No TEST rows for top station_id={top_station_id}")


## Bottom Station (Test)


In [ ]:
mask_bottom = X_test_eval["start_station_id"] == bottom_station_id

if mask_bottom.any():
    X_test_bottom = X_test_eval.loc[mask_bottom]
    y_test_bottom = y_true.loc[mask_bottom]
    y_test_pred_bottom = y_pred.loc[mask_bottom]

    compute_metrics(y_test_bottom, y_test_pred_bottom, "Bottom station (Test)", mask_bottom)

    plot_actual_vs_pred(
        X_test_bottom,
        y_test_bottom,
        y_test_pred_bottom,
        date_col="date",
        aggregate=True,
        title=f"Actual vs Predicted (Bottom station_id={bottom_station_id}, TEST)",
    )
else:
    print(f"No TEST rows for bottom station_id={bottom_station_id}")


## Holiday (Test)


In [ ]:
mask_holiday = X_test_eval["is_holiday"].astype(int).eq(1).reset_index(drop=True)

if mask_holiday.any():
    X_test_holiday = X_test_eval.loc[mask_holiday]
    y_true_holiday = y_true[mask_holiday]
    y_pred_holiday = y_pred[mask_holiday]

    compute_metrics(y_true_holiday, y_pred_holiday, "Holiday (Test)", mask_holiday)

    plot_actual_vs_pred(
        X_test_holiday,
        y_true_holiday,
        y_pred_holiday,
        date_col="date",
        aggregate=True,
        title="Actual vs Predicted (Holiday, TEST)",
    )
else:
    print("No holiday rows in TEST set.")


## Segment Night (00:00-05:59) - Test


In [ ]:
mask_night = X_test_eval["segment_id"].astype(int).eq(0).reset_index(drop=True)

if mask_night.any():
    X_test_night = X_test_eval.loc[mask_night]
    y_true_night = y_true[mask_night]
    y_pred_night = y_pred[mask_night]

    compute_metrics(y_true_night, y_pred_night, "Night segment (Test)", mask_night)

    plot_actual_vs_pred(
        X_test_night,
        y_true_night,
        y_pred_night,
        date_col="date",
        aggregate=True,
        title="Actual vs Predicted (Night segment, TEST)",
    )
else:
    print("No night-segment rows in TEST set.")


## Segment Day (06:00-15:59) - Test


In [ ]:
mask_day = X_test_eval["segment_id"].astype(int).eq(1).reset_index(drop=True)

if mask_day.any():
    X_test_day = X_test_eval.loc[mask_day]
    y_true_day = y_true[mask_day]
    y_pred_day = y_pred[mask_day]

    compute_metrics(y_true_day, y_pred_day, "Day segment (Test)", mask_day)

    plot_actual_vs_pred(
        X_test_day,
        y_true_day,
        y_pred_day,
        date_col="date",
        aggregate=True,
        title="Actual vs Predicted (Day segment, TEST)",
    )
else:
    print("No day-segment rows in TEST set.")


## Segment Evening (16:00-23:59) - Test


In [ ]:
mask_evening = X_test_eval["segment_id"].astype(int).eq(2).reset_index(drop=True)

if mask_evening.any():
    X_test_evening = X_test_eval.loc[mask_evening]
    y_true_evening = y_true[mask_evening]
    y_pred_evening = y_pred[mask_evening]

    compute_metrics(y_true_evening, y_pred_evening, "Evening segment (Test)", mask_evening)

    plot_actual_vs_pred(
        X_test_evening,
        y_true_evening,
        y_pred_evening,
        date_col="date",
        aggregate=True,
        title="Actual vs Predicted (Evening segment, TEST)",
    )
else:
    print("No evening-segment rows in TEST set.")


In [ ]:
# Save dual models + metadata
import joblib

model_dir = Path("models")
model_dir.mkdir(parents=True, exist_ok=True)

joblib.dump(best_model_high, model_dir / "catboost_station_3segments_high.joblib")
joblib.dump(best_model_low, model_dir / "catboost_station_3segments_low.joblib")

meta = {
    "feature_cols_cb": feature_cols_cb,
    "cat_cols": cat_cols,
    "high_stations": sorted([int(x) for x in high_stations]),
    "low_stations": sorted([int(x) for x in low_stations]),
    "volume_split_cum_share": 0.80,
}
joblib.dump(meta, model_dir / "catboost_station_3segments_dual_meta.joblib")

print("Saved:")
print(model_dir / "catboost_station_3segments_high.joblib")
print(model_dir / "catboost_station_3segments_low.joblib")
print(model_dir / "catboost_station_3segments_dual_meta.joblib")


 ```bash
 
 1) Charger modèles + meta
model_dir = Path("models")
model_high = joblib.load(model_dir / "catboost_station_3segments_high.joblib")
model_low = joblib.load(model_dir / "catboost_station_3segments_low.joblib")
meta = joblib.load(model_dir / "catboost_station_3segments_dual_meta.joblib")

feature_cols_cb = meta["feature_cols_cb"]
high_stations = set(meta["high_stations"])
low_stations = set(meta["low_stations"])

# 2) Préparer ton DataFrame d'inférence (X_new doit contenir les mêmes colonnes features)
# Exemple: X_new = df_future.copy()
X_inf = X_new[feature_cols_cb].copy()
X_inf["start_station_id"] = np.rint(X_inf["start_station_id"]).astype("int32")

# 3) Routage dual (high / low / unknown)
mask_high = X_inf["start_station_id"].astype(int).isin(high_stations).to_numpy()
mask_low = X_inf["start_station_id"].astype(int).isin(low_stations).to_numpy()
mask_unknown = ~(mask_high | mask_low)

y_pred = np.zeros(len(X_inf), dtype=float)

if mask_high.any():
    y_pred[mask_high] = model_high.predict(X_inf.loc[mask_high])

if mask_low.any():
    y_pred[mask_low] = model_low.predict(X_inf.loc[mask_low])

if mask_unknown.any():
    # fallback stations jamais vues au train
    y_pred[mask_unknown] = model_high.predict(X_inf.loc[mask_unknown])

# optionnel (demande non négative + arrondi)
y_pred_ops = np.rint(np.clip(y_pred, 0, None)).astype(int)

print("Predictions:", y_pred[:5])
print("Predictions ops:", y_pred_ops[:5])
